# 🎙️ Deepfake Detection Pipeline

**Steps:**
1. Securely Clone Repo
2. Install Dependencies
3. Download/Link Data
4. Run Audio Generation
5. Extract Features


## 1️⃣ Secure Setup & Clone

In [ ]:
# Mount Drive
from google.colab import drive
from getpass import getpass
import os
import shutil

drive.mount('/content/drive')

# --- CONFIGURATION ---
USERNAME = "charlesgrube-jpg"
REPO_NAME = "Data-Management-DDAA-KAN"
BRANCH = "feature/tts-vc-extraction"
# ---------------------

print("Enter your GitHub Personal Access Token (Permissions: repo):")
token = getpass()

# Secure URL construction
repo_url = f"https://{token}@github.com/{USERNAME}/{REPO_NAME}.git"

if not os.path.exists(REPO_NAME):
    !git clone {repo_url} {REPO_NAME}
    %cd {REPO_NAME}
    !git checkout {BRANCH}
    print(f"✅ Cloned and checked out {BRANCH}!")
else:
    %cd {REPO_NAME}
    print("✅ Repo already exists.")

In [ ]:
# Install Deps
!apt-get install -y ffmpeg espeak-ng

# --- CRITICAL FIX: FORCE INSTALLATION BYPASS ---
# 1. Install Fairseq (ignoring broken metadata)
!pip install --no-deps git+https://github.com/facebookresearch/fairseq.git
# 2. Install RVC (ignoring dependencies to prevent re-triggering Fairseq check)
!pip install --no-deps git+https://github.com/MisileLab/rvc-python-butter.git
# ------------------------------------------------

# Use %pip to ensure installation in current kernel
# This installs all dependencies for RVC/Fairseq that we skipped above
%pip install -r requirements_colab.txt

## 2️⃣ Data Setup

In [ ]:
from tqdm.notebook import tqdm
import os
import shutil
from glob import glob

# --- CONFIGURATION ---
# 1. Path to your 'en' folder in Drive (Find via file browser -> Copy Path)
CUSTOM_SOURCE_PATH = ""  # e.g. "/content/drive/Othercomputers/My Laptop/en"

# 2. Limit files for testing? (Set to 0 for ALL files)
MAX_FILES_TO_COPY = 50
# ---------------------

# Use absolute path to the REPO directory so data lands in the right place
TARGET_DIR = "/content/Data-Management-DDAA-KAN/mozilla_cv_data"

def smart_copy(src, dst):
    """Copies folder with progress bar and optional file limit"""
    print(f"📂 Scanning source: {src} ...")
    
    # Collect all files first
    all_files = []
    for root, dirs, files in os.walk(src):
        for file in files:
            all_files.append((os.path.join(root, file), os.path.relpath(os.path.join(root, file), src)))
            
    total_files = len(all_files)
    print(f"📊 Found {total_files} total files.")
    
    # Apply limit if set
    if MAX_FILES_TO_COPY > 0 and total_files > MAX_FILES_TO_COPY:
        # Prioritize TSV files so metadata is preserved!
        tsvs = [f for f in all_files if f[0].endswith('.tsv')]
        others = [f for f in all_files if not f[0].endswith('.tsv')]
        
        # Combine TSVs + a subset of other files
        keep_others = others[:MAX_FILES_TO_COPY]
        files_to_copy = tsvs + keep_others
        print(f"✂️ LIMITING copy to {len(files_to_copy)} files (TSVs + {MAX_FILES_TO_COPY} audio sample).")
    else:
        files_to_copy = all_files

    if os.path.exists(dst):
        shutil.rmtree(dst)
    os.makedirs(dst, exist_ok=True)

    print("🚀 Starting copy...")
    for src_path, rel_path in tqdm(files_to_copy, unit="file"):
        dst_path = os.path.join(dst, rel_path)
        os.makedirs(os.path.dirname(dst_path), exist_ok=True)
        shutil.copy2(src_path, dst_path)
        
    print("✅ Smart Copy Complete!")

def setup_data():
    if not CUSTOM_SOURCE_PATH or not os.path.exists(CUSTOM_SOURCE_PATH):
         print("⚠️ Please set CUSTOM_SOURCE_PATH above to your synced folder location!")
         return

    # Check if folder or file
    if os.path.isdir(CUSTOM_SOURCE_PATH):
        smart_copy(CUSTOM_SOURCE_PATH, TARGET_DIR)
    else:
        # Fallback for tar/zip (legacy support)
        print("📦 Detected archive file. Extracting...")
        if CUSTOM_SOURCE_PATH.endswith(".tar.gz"):
             !mkdir -p {TARGET_DIR}
             !tar -xzf "{CUSTOM_SOURCE_PATH}" -C {TARGET_DIR}
        elif CUSTOM_SOURCE_PATH.endswith(".zip"):
             !unzip -q "{CUSTOM_SOURCE_PATH}" -d {TARGET_DIR}
        print("✅ Extraction complete!")

setup_data()

## 3️⃣ Run Pipeline (Generate Audio)

In [ ]:
import yaml
import os

# Ensure we are in the repo directory
if os.path.exists("run_pipeline.py"):
    print("✅ In correct directory.")
else:
    print("⚠️ Wrong directory! Attempting to find repo...")
    if os.path.exists("Data-Management-DDAA-KAN"):
        %cd Data-Management-DDAA-KAN
        print("✅ Switched to repo directory.")
    else:
        print("❌ Could not find repo! Did you clone it?")

OUTPUT_PATH = "/content/drive/MyDrive/DDAA_Pipeline_Output"

with open("config.yaml", "r") as f:
    config = yaml.safe_load(f)

# Configure for Colab
config['output']['base_dir'] = OUTPUT_PATH
config['synthesis']['tts_models'] = []
config['codec_compression']['enabled'] = True
# FIX: Point config to the exact local copy path using the correct key
config['source']['data_path'] = "mozilla_cv_data"

with open("config.yaml", "w") as f:
    yaml.dump(config, f)

print("✅ Config updated to point to local data!")
!python run_pipeline.py

## 4️⃣ Extract Features

In [ ]:
import os
import glob

# 1. Find the latest output folder in Drive
drive_output_pattern = "/content/drive/MyDrive/DDAA_Pipeline_Output_*"
list_of_folders = glob.glob(drive_output_pattern)

if not list_of_folders:
    print("❌ No output folder found! Did the pipeline run successfully?")
else:
    latest_folder = max(list_of_folders, key=os.path.getctime)
    print(f"✅ Found latest dataset: {latest_folder}")
    
    # 2. Define output feature folder inside the dataset folder
    output_features = os.path.join(latest_folder, "features_cqt")

    print(f"🚀 Extracting CQT features to: {output_features}")
    
    # 3. Run Extraction
    !python -m pipeline.features.extract_features \
        --type cqt \
        --input "{latest_folder}" \
        --output "{output_features}"